## `multifactor-portfolio-analysis.ipynb`

---

# 1. Title & Project Overview

# Multifactor Portfolio Analysis Across Macroeconomic Regimes

This notebook explores the behavior of systematic multifactor portfolio strategies compared to a traditional 60/40 allocation model across changing macroeconomic environments.

The analysis focuses on:

- Risk-adjusted performance
- Drawdown resilience
- Portfolio adaptability during yield curve inversion periods

The project uses Python-based financial analysis and ETF portfolio construction.


---

# 2. Research Question

## Research Question

How do systematic multifactor portfolio strategies behave relative to traditional 60/40 allocation models under changing macroeconomic conditions?


---

# 3. Import Libraries

import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt


---

# 4. Load Data

This analysis combines:

- ETF historical price data retrieved using the yfinance library
- Macroeconomic data obtained from the Federal Reserve Economic Data (FRED) database

The datasets are used to evaluate portfolio behavior across changing macroeconomic conditions, including periods associated with U.S. Treasury yield curve inversion (10Y–2Y).

The download_etf_data.py script located in the /scripts directory contains the full ETF data acquisition and preprocessing workflow. For reference, the core download logic is shown below.


# --- Parameters ---
tickers = ["ACWI", "USMV", "QUAL", "MTUM", "AGG"]
start_date = "2005-01-01"
end_date = "2024-12-31"


# --- Download Data ---
data = yf.download(
    tickers,
    start=start_date,
    end=end_date,
    auto_adjust=False
)


# --- Validation ---
if data.empty:
    raise ValueError(
        "Downloaded data is empty. Check tickers or connection."
    )


print("Data downloaded successfully.")

Note: Additional data validation, alignment, and preprocessing logic is implemented in the download_etf_data.py script.


---

# 5. Initial Data Processing

Before constructing portfolios, the dataset is processed to ensure consistent structure, handle missing values, and compute monthly returns for portfolio analysis.

# --- Clean Data ---
df = df.sort_index()
df = df.dropna(how="all")

# --- Resample to Monthly ---
df_monthly = df.resample("ME").last()

# --- Calculate Returns ---
returns = df_monthly.pct_change().dropna()

# --- Enforce Column Order ---
returns = returns[["ACWI", "USMV", "QUAL", "MTUM", "AGG"]]

Note: Additional diagnostic and validation procedures are implemented within the preprocessing scripts located in the /scripts directory.


---

# 6. Portfolio Construction Logic

Three portfolio structures are evaluated throughout the analysis:

- Traditional 60/40 allocation
- Multifactor allocation
- Defensive allocation

The portfolio construction process incorporates quarterly rebalancing logic to account for portfolio drift and evolving asset weights throughout the investment horizon.

# --- Define Weights ---

# Traditional Portfolio
w_trad = {
    "ACWI": 0.60,
    "AGG": 0.40
}

# Multifactor Portfolio
w_mf = {
    "USMV": 0.20,
    "QUAL": 0.20,
    "MTUM": 0.20,
    "AGG": 0.40
}

# Defensive Portfolio
w_def = {
    "USMV": 0.30,
    "QUAL": 0.30,
    "AGG": 0.40
}




![](../output/portfolio_composition.png)


Portfolio returns are computed through a quarterly rebalancing framework that dynamically adjusts portfolio weights over time.

def compute_portfolio_returns(returns, weights):
    ...

The resulting portfolio return series are combined into a single DataFrame for comparative performance analysis.

portfolio_df = pd.DataFrame({
    "Traditional": trad_returns,
    "Multifactor": mf_returns,
    "Defensive": def_returns
})

Note: Additional portfolio construction, validation, and implementation details are available in the portfolio_construction.py script located in the /scripts directory.


---

# 7. Performance Metrics

Portfolio performance is evaluated using a set of risk-adjusted and drawdown-sensitive metrics designed to assess return efficiency, volatility behavior, and downside resilience across different macroeconomic environments.

The analysis incorporates:

- Compound Annual Growth Rate (CAGR)
- Annualized Volatility
- Sharpe Ratio
- Sortino Ratio
- Maximum Drawdown
- Cumulative Return

Together, these metrics provide a broader view of portfolio behavior beyond absolute returns alone.

def CAGR(returns):
    cumulative = (1 + returns).prod()
    n_periods = len(returns)
    return cumulative ** (periods_per_year / n_periods) - 1


def sharpe_ratio(returns):
    excess_returns = returns - rf_rate / periods_per_year
    vol = returns.std()

    if vol < epsilon:
        return np.nan

    return (excess_returns.mean() / vol) * np.sqrt(periods_per_year)


def max_drawdown(returns):
    cumulative = (1 + returns).cumprod()
    peak = cumulative.cummax()
    drawdown = (cumulative - peak) / peak

    return drawdown.min()



![](../output/cumulative_returns.png)



Note: Additional portfolio metric calculations and implementation details are available in the performance_metrics.py script located in the /scripts directory.


---

# 8. Regime-Based Performance Analysis

To evaluate portfolio behavior across different macroeconomic environments, portfolio returns are analyzed relative to the U.S. Treasury yield curve spread (10Y–2Y).

Periods are classified into two macroeconomic regimes:

- Normal yield curve environments
- Yield curve inversion environments

Portfolio performance metrics are then computed separately for each regime in order to evaluate differences in:

risk-adjusted performance,
volatility behavior,
and drawdown resilience.

# --- Classify Regime ---
yields["regime"] = np.where(
    yields["spread"] < 0,
    "Inversion",
    "Normal"
)


Note: Additional regime classification, portfolio aggregation, and performance evaluation logic are implemented within the macro_regime.py script located in the /scripts directory.


# 8. Visualization

To facilitate portfolio comparison across macroeconomic environments, the calculated performance metrics are organized into a comparative regime-based summary table.

The resulting structure allows for direct evaluation of portfolio behavior under both:

- normal yield curve environments
- yield curve inversion environments

The table summarizes differences in:

- return efficiency
- volatility behavior
- downside risk
- drawdown resilience across portfolio strategies


# --- Pivot Table ---
table = regime_df.pivot(index="Portfolio", columns="Regime")

# --- Organize Metrics ---
table = table[[
    ("CAGR", "Normal"), ("CAGR", "Inversion"),
    ("Volatility", "Normal"), ("Volatility", "Inversion"),
    ("Sharpe", "Normal"), ("Sharpe", "Inversion"),
    ("Sortino", "Normal"), ("Sortino", "Inversion"),
    ("Max Drawdown", "Normal"), ("Max Drawdown", "Inversion")
]]

# --- Round Values ---
table = table.round(3)

# --- Rename Columns ---
table.columns = [
    "CAGR (N)", "CAGR (I)",
    "Vol (N)", "Vol (I)",
    "Sharpe (N)", "Sharpe (I)",
    "Sortino (N)", "Sortino (I)",
    "MaxDD (N)", "MaxDD (I)"
]


# --- Display ---
table_pct = table.copy()



![](../output/regime_sharpe_comparison.png)




The resulting comparative framework serves as the foundation for evaluating the relative resilience and risk-adjusted performance of each portfolio structure across changing macroeconomic conditions.

Note: Additional visualization formatting and analytical reporting logic are implemented within the tabla_regimenes.py script located in the /scripts directory.


---

# 9. Initial Observations

- Multifactor portfolio structures generally produced stronger risk-adjusted performance relative to the traditional 60/40 allocation framework.

- The inclusion of Momentum contributed to higher portfolio returns, particularly during sustained market trends, although it also increased sensitivity to abrupt market shifts.

- Defensive allocations emphasizing Low Volatility and Quality exhibited lower volatility and more moderate drawdowns during stressed market environments.

- Portfolio behavior varied across macroeconomic regimes, suggesting that factor effectiveness may change depending on broader financial and economic conditions.

- The use of ETFs provides a practical approximation of factor exposures, although the results may differ from theoretical long-short factor constructions commonly referenced in academic literature.

- The findings suggest that systematic factor integration may enhance portfolio resilience and return efficiency under changing macroeconomic environments.


---

# 10. Limitations

This analysis presents several limitations that should be considered when interpreting the results.

- The study uses ETFs as practical proxies for factor exposure rather than theoretical long-short factor constructions commonly referenced in academic literature.

- Transaction costs, taxes, spreads, and other implementation frictions associated with portfolio rebalancing are not incorporated into the analysis.

- The macroeconomic framework is primarily based on the U.S. Treasury yield curve spread (10Y–2Y), which may not fully capture broader economic and financial regime dynamics.

- The portfolio universe maintains significant exposure to U.S. equity markets, limiting the generalizability of the findings across international and emerging market environments.

- Although the analysis includes multiple market conditions between 2013 and 2025, the historical horizon remains limited relative to the long-term complexity of financial markets.

---

# 11. Conclusion

The analysis suggests that the systematic incorporation of investment factors within a traditional 60/40 allocation framework may improve risk-adjusted portfolio performance relative to static market capitalization-based allocation approaches.

Across the analyzed period, multifactor portfolio structures generally exhibited stronger return efficiency and greater resilience during periods associated with yield curve inversion and heightened market volatility. In particular, exposures linked to Quality and Low Volatility demonstrated more stable behavior during stressed macroeconomic environments, while Momentum contributed to enhanced return potential during sustained market trends.

The findings also highlight the importance of evaluating portfolio construction through a macroeconomic and regime-sensitive perspective rather than relying exclusively on static long-term allocation assumptions.

Although the analysis remains exploratory and subject to several implementation limitations, the results suggest that practical factor-based ETF allocations may provide a viable framework for improving portfolio diversification and structural resilience across changing market conditions.

Future extensions of the project may incorporate broader factor universes, additional macroeconomic indicators, dynamic allocation methodologies, and implementation cost analysis to further evaluate multifactor portfolio behavior under real-world investment constraints.

